## Imports

In [6]:
import csv
import random
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
import pickle

from sklearn.ensemble import RandomForestClassifier

## Import

In [7]:
better_soccer_data = []
with open('Preprocessed-Soccer-Dataset.csv', newline='', encoding='utf-8') as data:
    read = csv.reader(data)
    for x in read:
        better_soccer_data.append(x)

print('Rows loaded (including header):', len(better_soccer_data))
print('Columns:', len(better_soccer_data[0]))
print('Headers:', better_soccer_data[0])

Rows loaded (including header): 42527
Columns: 37
Headers: ['Home_L3_Pts', 'Home_L3_Goals', 'Home_L3_EnemyGoals', 'Home_L3_Wins', 'Home_L3_Losses', 'Home_L3_HomeWins', 'Home_L5_Pts', 'Home_L5_Goals', 'Home_L5_EnemyGoals', 'Home_L5_Wins', 'Home_L5_Losses', 'Home_L5_HomeWins', 'Home_L10_Pts', 'Home_L10_Goals', 'Home_L10_EnemyGoals', 'Home_L10_Wins', 'Home_L10_Losses', 'Home_L10_HomeWins', 'Away_L3_Pts', 'Away_L3_Goals', 'Away_L3_EnemyGoals', 'Away_L3_Wins', 'Away_L3_Losses', 'Away_L3_HomeWins', 'Away_L5_Pts', 'Away_L5_Goals', 'Away_L5_EnemyGoals', 'Away_L5_Wins', 'Away_L5_Losses', 'Away_L5_HomeWins', 'Away_L10_Pts', 'Away_L10_Goals', 'Away_L10_EnemyGoals', 'Away_L10_Wins', 'Away_L10_Losses', 'Away_L10_HomeWins', 'Target_Did_Home_Win']


## Data Split

In [8]:
data = better_soccer_data[1:]  # skip header row

eighty = int((len(data)) * 0.8)
training_set = data[:eighty]
testing_set  = data[eighty:]

training_labels = []
for r in training_set: training_labels.append(r.pop())
testing_labels = []
for r in testing_set: testing_labels.append(r.pop())

# Convert to numbers (float)
for i in range(len(training_set)):
    for j in range(len(training_set[i])):
        training_set[i][j] = float(training_set[i][j])

for i in range(len(testing_set)):
    for j in range(len(testing_set[i])):
        testing_set[i][j] = float(testing_set[i][j])

print('Training rows:', len(training_set))
print('Testing rows: ', len(testing_set))
print('Features per row:', len(training_set[0]))

Training rows: 34020
Testing rows:  8506
Features per row: 36


# 1st Model: Logistic Regression (Full Fat Version)

Logistic Regression is a good baseline and is generally decent for binary classification.

In [9]:
model = LogisticRegression()
model.fit(training_set, training_labels)

pickle.dump(model, open('finalized_model_M1.sav', 'wb'))
print('Model 1 saved: finalized_model_M1.sav')
print('Trained on', model.n_features_in_, 'features')

Model 1 saved: finalized_model_M1.sav
Trained on 36 features


## 2nd Model: Reduced Data Set (Top 24 Features)
L3 did worse than L5 or L10. Sustained performance over more games is a stronger predictor of match outcome than short-term form.

Features:
- Home: Last 5 games (index 6–11) and Last 10 games (index 12–17)
- Away: Last 5 games (index 24–29) and Last 10 games (index 30–35)

In [10]:
# Indices of the top features by Pearson correlation (L5 and L10 windows only)
INDEX4reduced = [6, 7, 8, 9, 10, 11,    # Home L5
                 12, 13, 14, 15, 16, 17, # Home L10
                 24, 25, 26, 27, 28, 29, # Away L5
                 30, 31, 32, 33, 34, 35] # Away L10

# Build reduced training and testing sets
training_setR = []
for r in training_set:
    rowR = []
    for i in INDEX4reduced:
        rowR.append(r[i])
    training_setR.append(rowR)

testing_setR = []
for r in testing_set:
    rowR = []
    for i in INDEX4reduced:
        rowR.append(r[i])
    testing_setR.append(rowR)


# 2nd Model: Logistic Regression (Skim Milk)

In [11]:

model2 = LogisticRegression()
model2.fit(training_setR, training_labels)

pickle.dump(model2, open('finalized_model_M2.sav', 'wb'))
print('Model 2 saved: finalized_model_M2.sav')
print('Trained on', model2.n_features_in_, 'features')

Model 2 saved: finalized_model_M2.sav
Trained on 24 features


## Model: KNN (Full Feature Set)

In [12]:
knn_full = KNeighborsClassifier(n_neighbors=15)
knn_full.fit(training_set, training_labels)
score = knn_full.score(testing_set, testing_labels)
print(f'KNN(k=15): {round(score * 100, 2)}%')

pickle.dump(knn_full, open('knn_full.sav', 'wb'))
print('KNN Model on Full Feature Set saved: knn_full.sav')
print('Trained on', knn_full.n_features_in_, 'features')

KNN(k=15): 57.02%
KNN Model on Full Feature Set saved: knn_full.sav
Trained on 36 features


## Model : Gaussian Naive Bayes (Full Feature Set)

In [13]:
naive_full = GaussianNB()
naive_full.fit(training_set, training_labels)
score = naive_full.score(testing_set, testing_labels)
print(f'Gaussian Naive Bayes: {round(score * 100, 2)}%')

pickle.dump(naive_full, open('naive_bayes_full.sav', 'wb'))
print('Gaussian Naive Bayes Model on Full Feature Set saved: naive_bayes_full.sav')
print('Trained on', naive_full.n_features_in_, 'features')

Gaussian Naive Bayes: 60.99%
Gaussian Naive Bayes Model on Full Feature Set saved: naive_bayes_full.sav
Trained on 36 features


## Model: Random Forest (Full Feature Set)

In [ ]:
randomTrees = RandomForestClassifier(n_estimators=300,max_depth=10,min_samples_split=5,random_state=101)
randomTrees.fit(training_set, training_labels)

resultTrees = randomTrees.score(testing_set, testing_labels)

pickle.dump(randomTrees, open('randomTrees.sav', 'wb'))
print("Random Forest (Full features) score: " + str(round(resultTrees * 100, 2)) + "%")
print(randomTrees.feature_importances_)

Random Forest (Full features) score: 61.62%
[0.01692272 0.02430865 0.02277149 0.00761815 0.00830706 0.00692223
 0.02811546 0.03602115 0.02687097 0.01308782 0.01338504 0.01015943
 0.07284673 0.0823365  0.04272446 0.04516077 0.03793968 0.01797395
 0.01652624 0.02831863 0.02420627 0.00812508 0.00838483 0.00762464
 0.02827255 0.03738139 0.02840558 0.01281984 0.01207055 0.00971144
 0.06095812 0.06886975 0.0446514  0.03664076 0.03555474 0.01800592]


## # Load Models from disk & print()

In [7]:
load1 = pickle.load(open('finalized_model_M1.sav', 'rb'))
load2 = pickle.load(open('finalized_model_M2.sav', 'rb'))

result1 = load1.score(testing_set, testing_labels)
result2 = load2.score(testing_setR, testing_labels)

print('Model 1 (Full features) score:    ' + str(round(result1 * 100, 2)) + '%')
print('Model 2 (Reduced features) score: ' + str(round(result2 * 100, 2)) + '%')
	# Looks like we get about 60% accuracy for both
	# This means it's learning something from the data, just not much XD

Model 1 (Full features) score:    62.16%
Model 2 (Reduced features) score: 62.12%
